In [1]:
# (DO NOT containerize this cell)

# Ville dont la prévision horaire alimente le graphique xy-plot.
param_city = 'Amsterdam'

# Nombre d'heures de prévision à tracer.
param_forecast_hours = '48'

In [2]:
# (DO NOT containerize this cell)

# Workdir partagé entre les pods d'un même run (emptyDir monté en /tmp/data,
# cf. NaaVRE-workflow-service/app/templates/argo_workflowSpec.j2).
conf_local_tmp = '/tmp/data'

# Coordonnées des villes (lat, lon).
conf_cities = {
    'Amsterdam': (52.37, 4.89),
    'Paris': (48.85, 2.35),
    'Berlin': (52.52, 13.41),
    'Rome': (41.90, 12.50),
    'Lisbon': (38.72, -9.14),
}

In [3]:
# S1 Fetch weather
# base image: vanilla

import json
import os
from urllib import request

def fetch_json(url):
    with request.urlopen(url, timeout=15) as resp:
        return json.loads(resp.read().decode('utf-8'))

def fetch_current(lat, lon):
    url = (
        'https://api.open-meteo.com/v1/forecast'
        f'?latitude={lat}&longitude={lon}'
        '&current=temperature_2m,relative_humidity_2m,wind_speed_10m'
        '&timezone=auto'
        )
    return fetch_json(url)['current']

def fetch_hourly(lat, lon, hours):
    url = (
        'https://api.open-meteo.com/v1/forecast'
        f'?latitude={lat}&longitude={lon}'
        '&hourly=temperature_2m&forecast_days=2&timezone=auto'
        )
    return fetch_json(url)['hourly']['temperature_2m'][:hours]

os.makedirs(conf_local_tmp, exist_ok=True)

# --- forecast.csv (xy-plot) : prévision horaire de param_city ---
lat, lon = conf_cities[param_city]
hourly_temps = fetch_hourly(lat, lon, int(param_forecast_hours))
forecast_path = os.path.join(conf_local_tmp, 'forecast.csv')
with open(forecast_path, 'w', encoding='utf-8') as f:
    f.write('hour,temperature_c\n')
    for i, t in enumerate(hourly_temps):
        f.write(f'{i},{t}\n')
print(f'Wrote {forecast_path} ({len(hourly_temps)} rows)')

# --- cities.geojson (map) + summary.csv (table) : météo courante ---
features = []
for city, (city_lat, city_lon) in conf_cities.items():
    current = fetch_current(city_lat, city_lon)
    features.append({
        'type': 'Feature',
        'properties': {
            'name': city,
            'temperature_c': current['temperature_2m'],
            'humidity_pct': current['relative_humidity_2m'],
            'wind_speed_kmh': current['wind_speed_10m'],
            },
        'geometry': {'type': 'Point', 'coordinates': [city_lon, city_lat]},
        })

cities_geojson_path = os.path.join(conf_local_tmp, 'cities.geojson')
with open(cities_geojson_path, 'w', encoding='utf-8') as f:
    json.dump({'type': 'FeatureCollection', 'features': features}, f)
print(f'Wrote {cities_geojson_path} ({len(features)} features)')

summary_path = os.path.join(conf_local_tmp, 'summary.csv')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('city,temperature_c,humidity_pct,wind_speed_kmh\n')
    for feat in features:
        p = feat['properties']
        f.write(f"{p['name']},{p['temperature_c']},{p['humidity_pct']},{p['wind_speed_kmh']}\n")
print(f'Wrote {summary_path}')

Wrote /tmp/data/forecast.csv (48 rows)
Wrote /tmp/data/cities.geojson (5 features)
Wrote /tmp/data/summary.csv


In [4]:
!pip install ipywidgets

In [5]:
# S2 Weather widget
# base image: vanilla

import json
import os
from urllib import request

def fetch_json(url):
    with request.urlopen(url, timeout=15) as resp:
        return json.loads(resp.read().decode('utf-8'))

def fetch_current(lat, lon):
    url = (
        'https://api.open-meteo.com/v1/forecast'
        f'?latitude={lat}&longitude={lon}'
        '&current=temperature_2m,relative_humidity_2m,wind_speed_10m'
        '&timezone=auto'
        )
    return fetch_json(url)['current']

os.makedirs(conf_local_tmp, exist_ok=True)

tabs_html = []
for city, (lat, lon) in conf_cities.items():
    current = fetch_current(lat, lon)
    tabs_html.append(
        f"<section><h2>{city}</h2>"
        f"<p><b>{current['temperature_2m']} °C</b><br>"
        f"Humidity: {current['relative_humidity_2m']}%<br>"
        f"Wind: {current['wind_speed_10m']} km/h</p></section>"
        )

page = (
    "<!DOCTYPE html><html lang='en'><head><meta charset='UTF-8'>"
    "<title>City weather</title><style>"
    "body{font-family:sans-serif;display:flex;gap:1rem;flex-wrap:wrap}"
    "section{border:1px solid #ccc;border-radius:8px;padding:1rem}"
    "</style></head><body>" + "".join(tabs_html) + "</body></html>"
    )

widget_html_path = os.path.join(conf_local_tmp, 'weather_widget.html')
with open(widget_html_path, 'w') as f:
    f.write(page)

print(f'Wrote {widget_html_path}')

Wrote /tmp/data/weather_widget.html


In [6]:
# (DO NOT containerize this cell)

for path in [forecast_path, cities_geojson_path, summary_path, widget_html_path]:
    print(path, '-', os.path.getsize(path), 'bytes')

with open(summary_path) as f:
    print()
    print(f.read())

/tmp/data/forecast.csv - 393 bytes
/tmp/data/cities.geojson - 957 bytes
/tmp/data/summary.csv - 145 bytes
/tmp/data/weather_widget.html - 712 bytes

city,temperature_c,humidity_pct,wind_speed_kmh
Amsterdam,18.6,80,7.9
Paris,20.4,62,12.0
Berlin,17.7,68,17.2
Rome,28.4,60,11.6
Lisbon,22.9,64,3.8

